# Z-Image-Turbo｜圖片生成（Colab T4）

設定 Prompt、Seed、尺寸後逐格執行，即可直接預覽並下載 PNG。 ComfyUI 僅在 Colab 內部的 `127.0.0.1` 運作，不會建立 ngrok 或對外服務。

**建議每個 Notebook 使用新的 Colab 執行階段**，避免不同模型安裝的套件互相衝突。

## 1. 安裝所需套件（和 Python import 分開）

In [ ]:
# Install ComfyUI for image generation.
# 安裝圖片生成所需的 ComfyUI。
!test -d /content/ComfyUI || git clone -q https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI

# Install GGUF support for quantized models.
# 安裝量化模型所需的 GGUF 支援。
!test -d /content/ComfyUI/custom_nodes/ComfyUI-GGUF || git clone -q https://github.com/city96/ComfyUI-GGUF.git /content/ComfyUI/custom_nodes/ComfyUI-GGUF

# Install ComfyUI packages.
# 安裝 ComfyUI 套件。
%pip -q install -r /content/ComfyUI/requirements.txt

# Install GGUF extension dependencies.
# 安裝 GGUF 擴充功能所需套件。
%pip -q install -r /content/ComfyUI/custom_nodes/ComfyUI-GGUF/requirements.txt

# Install model download and HTTP tools.
# 安裝模型下載與 HTTP 工具。
%pip -q install -U huggingface_hub hf_xet requests

## 2. 匯入模組

In [ ]:
# Import standard Python tools.
# 匯入 Python 標準工具。
import subprocess
import sys
import time
import uuid
from pathlib import Path

# Import model download and communication tools.
# 匯入模型下載與通訊工具。
import requests
import torch
from huggingface_hub import hf_hub_download

# Import image display and Colab download tools.
# 匯入圖片顯示與 Colab 下載工具。
from IPython.display import display
from PIL import Image
from google.colab import files

## 3. 修改生成參數

**PROMPT**：想看到什麼；**SEED**：隨機起點；**WIDTH／HEIGHT**：圖片尺寸（建議為 64 的倍數）。此 Turbo GGUF 工作流程預設 **STEPS=8、CFG=1**，先不用調整。

In [ ]:
# Set the image description.
# 設定圖片內容描述。
PROMPT = "A tabby cat by a cafe window, warm afternoon light, realistic photography"

# Choose output size and random seed.
# 設定圖片尺寸與隨機種子。
WIDTH, HEIGHT, SEED = 576, 768, 42

# Keep the recommended Turbo sampling settings.
# 保留 Turbo 建議的取樣設定。
STEPS, CFG = 8, 1

# Check that a Colab GPU is enabled.
# 確認 Colab 已啟用 GPU。
assert torch.cuda.is_available(), "請先將 Colab 執行階段切換為 T4 GPU。"
print(torch.cuda.get_device_name(0))

## 4. 下載模型（首次執行會較久）

In [ ]:
# List the three required model files.
# 列出需要下載的三個模型檔案。
model_files = [
    ("unsloth/Z-Image-Turbo-GGUF", "z-image-turbo-Q6_K.gguf", "unet/z-image-turbo-Q6_K.gguf"),
    ("Comfy-Org/z_image_turbo", "split_files/text_encoders/qwen_3_4b_fp8_mixed.safetensors", "text_encoders/qwen_3_4b_fp8_mixed.safetensors"),
    ("Comfy-Org/z_image_turbo", "split_files/vae/ae.safetensors", "vae/ae.safetensors"),
]

# Download once and link each file to ComfyUI.
# 下載一次，並將模型檔案連結到 ComfyUI。
for repo, filename, target in model_files:
    destination = Path("/content/ComfyUI/models") / target
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.is_file():
        saved = hf_hub_download(repo_id=repo, filename=filename)
        destination.symlink_to(saved)
    print("Ready:", destination.name)

## 5. 準備生成流程（不需要修改）

這段將模型、Prompt、取樣與圖片輸出接起來。與原本的 ComfyUI 工作流程相同，但**不用手動編輯 JSON**。

In [ ]:
# Define a minimal ComfyUI workflow for Z-Image-Turbo.
# 建立精簡的 Z-Image-Turbo ComfyUI 工作流程。
workflow = {
    "1": {"class_type": "UnetLoaderGGUF", "inputs": {"unet_name": "z-image-turbo-Q6_K.gguf"}},
    "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": "qwen_3_4b_fp8_mixed.safetensors", "type": "lumina2", "device": "default"}},
    "3": {"class_type": "VAELoader", "inputs": {"vae_name": "ae.safetensors"}},
    "4": {"class_type": "CLIPTextEncode", "inputs": {"text": PROMPT, "clip": ["2", 0]}},
    "5": {"class_type": "ConditioningZeroOut", "inputs": {"conditioning": ["4", 0]}},
    "6": {"class_type": "EmptySD3LatentImage", "inputs": {"width": WIDTH, "height": HEIGHT, "batch_size": 1}},
    "7": {"class_type": "ModelSamplingAuraFlow", "inputs": {"shift": 3, "model": ["1", 0]}},
    "8": {"class_type": "KSampler", "inputs": {"seed": SEED, "steps": STEPS, "cfg": CFG, "sampler_name": "res_multistep", "scheduler": "simple", "denoise": 1, "model": ["7", 0], "positive": ["4", 0], "negative": ["5", 0], "latent_image": ["6", 0]}},
    "9": {"class_type": "VAEDecode", "inputs": {"samples": ["8", 0], "vae": ["3", 0]}},
    "10": {"class_type": "SaveImage", "inputs": {"filename_prefix": "zimage", "images": ["9", 0]}},
}

# Set the internal address of ComfyUI.
# 設定 ComfyUI 在 Colab 內部的連線位址。
BASE = "http://127.0.0.1:8188"

## 6. 啟動模型服務（僅在 Colab 內使用）

In [ ]:
# Start ComfyUI only when it is not running.
# 只有在 ComfyUI 尚未啟動時才啟動。
try:
    running = requests.get(BASE + "/system_stats", timeout=2).ok
except requests.RequestException:
    running = False

# Run ComfyUI in the background with T4-friendly settings.
# 在背景啟動 ComfyUI，並使用較省 GPU 記憶體的設定。
if not running:
    log = Path("/content/zimage_server.log").open("w")
    server = subprocess.Popen([sys.executable, "/content/ComfyUI/main.py", "--listen", "127.0.0.1", "--port", "8188", "--lowvram", "--cpu-vae", "--cache-none", "--preview-method", "none"], cwd="/content/ComfyUI", stdout=log, stderr=subprocess.STDOUT)

# Wait until ComfyUI is ready.
# 等待 ComfyUI 準備完成。
for attempt in range(120):
    try:
        if requests.get(BASE + "/system_stats", timeout=2).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise RuntimeError("ComfyUI 無法啟動，請檢查 /content/zimage_server.log。")

# Show the connection status.
# 顯示連線狀態。
# Verify that the GGUF model loader was registered successfully.
# 確認 GGUF 模型載入節點已成功註冊。
response = requests.get(BASE + "/object_info/UnetLoaderGGUF", timeout=30)
response.raise_for_status()
assert "UnetLoaderGGUF" in response.json(), "無法載入 GGUF 節點，請檢查 ComfyUI 日誌。"
print("ComfyUI + GGUF ready")

## 7. 生成圖片

In [ ]:
# Submit the current image settings.
# 傳送目前設定，開始生成圖片。
response = requests.post(BASE + "/prompt", json={"prompt": workflow, "client_id": str(uuid.uuid4())}, timeout=120)

# Stop immediately if the workflow is invalid.
# 若工作流程有錯誤，立即顯示原因。
if not response.ok:
    raise RuntimeError(response.text)

# Save the generation job ID.
# 記住這次生成工作的編號。
job_id = response.json()["prompt_id"]
print("Generating:", job_id)

## 8. 預覽與下載 PNG

In [ ]:
# Wait until the generated image is available.
# 等待生成的圖片完成。
for attempt in range(900):
    record = requests.get(BASE + "/history/" + job_id, timeout=30).json().get(job_id)
    if record:
        if record.get("status", {}).get("status_str") == "error":
            raise RuntimeError(str(record["status"]))
        if record.get("outputs", {}).get("10", {}).get("images"):
            break
    time.sleep(2)
else:
    raise TimeoutError("等待圖片逾時，請檢查 ComfyUI 紀錄。")

# Save the completed PNG file.
# 儲存完成的 PNG 圖片。
image_info = record["outputs"]["10"]["images"][0]
image_response = requests.get(BASE + "/view", params=image_info, timeout=120)
image_response.raise_for_status()
image_path = Path("/content/zimage_output.png")
image_path.write_bytes(image_response.content)

# Preview the image and download it locally.
# 顯示圖片，並下載到自己的電腦。
display(Image.open(image_path))
files.download(str(image_path))